In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision as tv

from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import cv2
import numpy as np
from PIL import ImageFile
from google.colab import drive
from google.colab import files

In [18]:
drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/kagglecatsanddogs_3367a/PetImages" /content/

data_dir = "/content/PetImages"

print(os.listdir(data_dir))  # ['Cat', 'Dog']

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Dog', 'Cat']


In [19]:
trans = tv.transforms.Compose([
    tv.transforms.Resize((128, 128)),
    tv.transforms.RandomHorizontalFlip(),
    tv.transforms.RandomRotation(10),
    tv.transforms.ToTensor(),
    tv.transforms.Normalize([0.5]*3, [0.5]*3)
])

In [20]:
dataset = tv.datasets.ImageFolder(
    data_dir,
    transform=trans
)

In [21]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

In [22]:
batch_size = 64

train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = torch.utils.data.DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [23]:
class CatsDogsCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 100),
            nn.ReLU(),
            nn.Linear(100, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = CatsDogsCNN().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

device: cuda


In [25]:
def accuracy(pred, label):
    pred_classes = pred.argmax(dim=1)
    return (pred_classes == label).float().mean().item()

In [26]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()

    total_loss = 0
    total_acc = 0

    for img, label in (pbar := tqdm(loader)):
        img, label = img.to(device), label.to(device)

        optimizer.zero_grad()

        pred = model(img)
        loss = loss_fn(pred, label)

        loss.backward()
        optimizer.step()

        loss_item = loss.item()
        acc = accuracy(pred, label)

        total_loss += loss_item
        total_acc += acc

        pbar.set_description(f'loss: {loss_item:.4f} acc: {acc:.3f}')

    return total_loss / len(loader), total_acc / len(loader)


In [27]:
def validate(model, loader, loss_fn):
    model.eval()

    total_loss = 0
    total_acc = 0

    with torch.no_grad():
        for img, label in loader:
            img, label = img.to(device), label.to(device)

            pred = model(img)
            loss = loss_fn(pred, label)

            total_loss += loss.item()
            total_acc += accuracy(pred, label)

    return total_loss / len(loader), total_acc / len(loader)

In [28]:
epochs = 5

train_losses = []
val_losses = []

train_accs = []
val_accs = []

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, loss_fn, optimizer
    )

    val_loss, val_acc = validate(
        model, val_loader, loss_fn
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

  0%|          | 0/312 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
loss: 0.5569 acc: 0.750:  99%|█████████▊| 308/312 [00:54<00:00,  6.40it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
loss: 0.7166 acc: 0.667: 100%|██████████| 312/312 [00:54<00:00,  5.70it/s]



Epoch 1
Train Loss: 0.5991 | Train Acc: 0.6678
Val   Loss: 0.5098 | Val   Acc: 0.7498


loss: 0.5500 acc: 0.734:  27%|██▋       | 85/312 [00:15<00:33,  6.76it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
loss: 0.4018 acc: 0.810: 100%|██████████| 312/312 [00:56<00:00,  5.50it/s]



Epoch 2
Train Loss: 0.4873 | Train Acc: 0.7608
Val   Loss: 0.4556 | Val   Acc: 0.7883


loss: 0.3941 acc: 0.797:  32%|███▏      | 101/312 [00:18<00:33,  6.30it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
loss: 0.4540 acc: 0.778: 100%|██████████| 312/312 [00:54<00:00,  5.75it/s]



Epoch 3
Train Loss: 0.4325 | Train Acc: 0.7989
Val   Loss: 0.4137 | Val   Acc: 0.8171


loss: 0.3570 acc: 0.859:  92%|█████████▏| 286/312 [00:52<00:05,  4.66it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
loss: 0.3749 acc: 0.810: 100%|██████████| 312/312 [00:56<00:00,  5.49it/s]



Epoch 4
Train Loss: 0.3979 | Train Acc: 0.8198
Val   Loss: 0.3915 | Val   Acc: 0.8263


loss: 0.3572 acc: 0.812:  54%|█████▍    | 168/312 [00:30<00:23,  6.09it/s]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
loss: 0.3792 acc: 0.857: 100%|██████████| 312/312 [00:54<00:00,  5.76it/s]



Epoch 5
Train Loss: 0.3704 | Train Acc: 0.8325
Val   Loss: 0.3527 | Val   Acc: 0.8389


# Тесты

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(filename)

In [ ]:
img = cv2.imread(filename)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (128, 128))
img = img / 255.0

img = torch.tensor(img, dtype=torch.float32)

img = img.permute(2, 0, 1)

img = img.unsqueeze(0)

img = img.to(device)

In [ ]:
model.eval()

with torch.no_grad():
    pred = model(img)
    pred_class = pred.argmax(dim=1).item()

print("Prediction:", pred_class)

In [ ]:
classes = dataset.classes
print("Prediction:", classes[pred_class])

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(img.cpu()[0].permute(1, 2, 0))
plt.title(classes[pred_class])
plt.axis('off')
plt.show()